In [ ]:
import os
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import BaggingRegressor, ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import BayesianRidge, ElasticNet, HuberRegressor, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor


In [ ]:
pipelines = []
seed = 2

pipelines.append(
                ("Scaled_Lasso",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("Lasso", Lasso(random_state=seed))
                      ]))
                )

pipelines.append(
                ("Scaled_Elastic",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("Elastic", ElasticNet(random_state=seed))
                      ]))
                )


pipelines.append(
                ("Scaled_RF_reg",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("RF", RandomForestRegressor(random_state=seed))
                 ])
                )
                )

pipelines.append(
                ("Scaled_ET_reg",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("ET", ExtraTreesRegressor(random_state=seed))
                 ])
                )
                )
pipelines.append(
                ("Scaled_BR_reg",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("BR", BaggingRegressor(random_state=seed))
                 ])))

pipelines.append(
                ("Scaled_DT_reg",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("DT_reg", DecisionTreeRegressor(random_state=seed))
                 ])))

pipelines.append(
                ("Scaled_Ridge",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("Ridge", Ridge(random_state=seed))
                      ]))
                )


pipelines.append(
                ("Scaled_SVR",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("SVR",  SVR(kernel='linear', C=1e2))
                 ])
                )
                )

pipelines.append(
                ("Scaled_Hub-Reg",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("Hub-Reg", HuberRegressor())
                 ])))
pipelines.append(
                ("Scaled_BayRidge",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("BR", BayesianRidge())
                 ])))


pipelines.append(
                ("Scaled_KNN_reg",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("KNN_reg", KNeighborsRegressor())
                 ])))
pipelines.append(
                ("Scaled_Gboost-Reg",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("GBoost-Reg", GradientBoostingRegressor(random_state=seed))
                 ])))

pipelines.append(
                ("Scaled_XGB_reg",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("XGBR", XGBRegressor(random_state=seed))
                 ])))

pipelines.append(
                ("Scaled_RFR_PCA",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("PCA", PCA(n_components=3)),
                     ("RF", RandomForestRegressor(random_state=seed))
                 ])))

pipelines.append(
                ("Scaled_XGBR_PCA",
                 Pipeline([
                     ("Scaler", StandardScaler()),
                     ("PCA", PCA(n_components=3)),
                     ("XGB", XGBRegressor(random_state=seed))
                 ])))

#'neg_mean_absolute_error', 'neg_mean_squared_error','r2'
scoring = 'neg_mean_absolute_error'
n_folds = 7
results = {}


In [ ]:
DURATION_CAP = 960
target_col = "duration_hours"
DATA_PATH_ENV = "DATA_FINALL_PATH"

data_path_value = os.environ.get(DATA_PATH_ENV)

DATA_PATH = Path(data_path_value).expanduser()

data_finall = pd.read_csv(DATA_PATH)

split = int(len(data_finall) * 0.8)
train_full = data_finall.iloc[:split].copy()
test_full  = data_finall.iloc[split:].copy()

train_filt   = train_full[train_full[target_col] < DURATION_CAP].copy()
test_typical = test_full[test_full[target_col] < DURATION_CAP].copy()

Xtrain = train_filt.drop(columns=[target_col], errors="ignore")
ytrain = train_filt[target_col]

Xtest_typical = test_typical.drop(columns=[target_col], errors="ignore")
ytest_typical = test_typical[target_col]

Xtest_full = test_full.drop(columns=[target_col], errors="ignore")
ytest_full = test_full[target_col]

# aliases for downstream plots: primary evaluation uses only typical tasks
Xtest = Xtest_typical.copy()
ytest = ytest_typical.copy()

print(f"DATA_PATH: {DATA_PATH}")
print(f"Всего строк:                    {len(data_finall)}")
print(f"Train raw:                      {len(train_full)}")
print(f"Train filtered < {DURATION_CAP}:           {len(train_filt)}")
print(f"Test typical < {DURATION_CAP}:            {len(test_typical)}")
print(f"Test full:                      {len(test_full)}")
print(f"Train outliers >= {DURATION_CAP}:         {(train_full[target_col] >= DURATION_CAP).sum()}")
print(f"Test outliers >= {DURATION_CAP}:          {(test_full[target_col] >= DURATION_CAP).sum()}")


In [ ]:
# Опциональная проверка чувствительности к порогу cap.
# Важно: этот блок НЕ используется для выбора финального cap.
# cap=960 зафиксирован предметной областью; здесь только локальная проверка устойчивости
# на типовых задачах holdout (`y < cap`).

RUN_CAP_SENSITIVITY = False

if not RUN_CAP_SENSITIVITY:
    print("CAP sensitivity skipped. cap=960 остаётся фиксированным доменным правилом.")
else:
    results = {}

    for cap in list(range(720, 1201, 60)):
        print(f"Computing cap={cap}")
        results[cap] = {}

        train_cap = train_full[train_full[target_col] < cap].copy()
        test_cap  = test_full[test_full[target_col] < cap].copy()

        Xtr = train_cap.drop(columns=[target_col], errors="ignore")
        ytr = train_cap[target_col]
        Xte = test_cap.drop(columns=[target_col], errors="ignore")
        yte = test_cap[target_col]

        for name, model in pipelines:
            est = clone(model)
            est.fit(Xtr, ytr)
            pred = est.predict(Xte)
            mae = mean_absolute_error(yte, pred)
            results[cap][name] = mae

    caps = sorted(results.keys())
    model_names = list(next(iter(results.values())).keys())

    plt.figure(figsize=(12, 6))
    for name in model_names:
        vals = [results[cap][name] / ytest_typical.mean() for cap in caps]
        plt.plot(caps, vals, label=name)

    plt.axvline(DURATION_CAP, linestyle="--", linewidth=2, color="black", label=f"fixed cap={DURATION_CAP}")
    plt.xlabel("Порог cap (только контрольная проверка, не тюнинг)")
    plt.ylabel("MAE / mean(ytest_typical)")
    plt.title("Чувствительность MAE к cap на типовых holdout-задачах")
    plt.legend(loc=7)
    plt.tight_layout()
    plt.show()


In [ ]:
# Второй старый sweep по cap удалён из финальной логики, чтобы не создавать впечатление,
# что cap подбирался по holdout. При необходимости оставлен только один опциональный
# контрольная проверка в предыдущей ячейке.

print("Block intentionally disabled: cap не подбирается по holdout, а фиксируется доменно.")


In [ ]:
# --- PRIMARY HOLDOUT + SECONDARY STRESS TEST ---
# Основная проверка: типовые задачи y < cap
# Дополнительная проверка: полный holdout, включая выбросы

art_dir = Path("./artifacts_baseline_pointfix")
art_dir.mkdir(parents=True, exist_ok=True)

def reg_metrics(y_true, pred):
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "MdAE": median_absolute_error(y_true, pred),
    }

print("=" * 100)
print("PRIMARY HOLDOUT (train < 960  ->  test_typical < 960)")
print("=" * 100)
print(f"  {'Model':<25s}  {'MAE':>8s}  {'RMSE':>8s}  {'MdAE':>8s}")
print("-" * 100)

results = {}
holdout_rows = []

for name, model in pipelines:
    est = clone(model)
    est.fit(Xtrain, ytrain)

    pred_typical = est.predict(Xtest_typical)
    pred_full = est.predict(Xtest_full)

    m_typ = reg_metrics(ytest_typical, pred_typical)
    m_full = reg_metrics(ytest_full, pred_full)

    results[name] = m_typ["MAE"]
    holdout_rows.append({
        "Model": name,
        "MAE_typical": m_typ["MAE"],
        "RMSE_typical": m_typ["RMSE"],
        "MdAE_typical": m_typ["MdAE"],
        "MAE_full": m_full["MAE"],
        "RMSE_full": m_full["RMSE"],
        "MdAE_full": m_full["MdAE"],
    })

    print(f"  {name:<25s}  {m_typ['MAE']:>8.2f}  {m_typ['RMSE']:>8.2f}  {m_typ['MdAE']:>8.2f}")

primary_holdout_summary = (
    pd.DataFrame(holdout_rows)
    .sort_values(["MAE_typical", "RMSE_typical", "MdAE_typical"])
    .reset_index(drop=True)
)

stress_holdout_summary = (
    primary_holdout_summary[["Model", "MAE_full", "RMSE_full", "MdAE_full"]]
    .sort_values(["MAE_full", "RMSE_full", "MdAE_full"])
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("WALK-FORWARD CV (только типовые задачи train<960 и valid<960)")
print("=" * 100)

tscv = TimeSeriesSplit(n_splits=3)
cv_results = {}

for name, model in pipelines:
    fold_maes = []

    for fold_num, (tr_idx, va_idx) in enumerate(tscv.split(train_filt), start=1):
        fold_train = train_filt.iloc[tr_idx].copy()
        fold_valid = train_filt.iloc[va_idx].copy()

        Xtr = fold_train.drop(columns=[target_col], errors="ignore")
        ytr = fold_train[target_col]

        Xva = fold_valid.drop(columns=[target_col], errors="ignore")
        yva = fold_valid[target_col]

        est = clone(model)
        est.fit(Xtr, ytr)
        pred = est.predict(Xva)

        fold_mae = mean_absolute_error(yva, pred)
        fold_maes.append(fold_mae)

    cv_results[name] = np.array(fold_maes)

    print(f"  {name:<25s}  MAE = {np.mean(fold_maes):.2f} +/- {np.std(fold_maes):.2f}")

cv_summary = (
    pd.DataFrame({
        "Model": list(cv_results.keys()),
        "CV_MAE_mean": [float(np.mean(v)) for v in cv_results.values()],
        "CV_MAE_std": [float(np.std(v)) for v in cv_results.values()],
    })
    .sort_values(["CV_MAE_mean", "CV_MAE_std"])
    .reset_index(drop=True)
)

primary_holdout_summary.to_csv(art_dir / "primary_holdout_summary.csv", index=False)
stress_holdout_summary.to_csv(art_dir / "stress_holdout_summary.csv", index=False)
cv_summary.to_csv(art_dir / "cv_summary.csv", index=False)

print("\nTop models on primary holdout:")
primary_holdout_summary.head(10)


In [ ]:
# cv_results: dict name -> array of 5 CV scores (neg MAE)
scores_df = pd.DataFrame({k: np.array(v) for k, v in cv_results.items()})

plt.figure(figsize=(16, 6), dpi=160)
sns.boxplot(data=scores_df)
plt.title("Сравнение моделей (TimeSeriesSplit): MAE по фолдам")
plt.xlabel("Model")
plt.ylabel("MAE  (меньше = лучше)")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Кривые обучения: Xtest фиксирован, меняем только размер Xtrain
plot_values = list(range(10, 100, 10))  # 10..90% от Xtrain
ncols = 2
nrows = math.ceil(len(plot_values) / ncols)
tail_n = 200

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 28))
axes = axes.flatten()

for idx, pct in enumerate(plot_values):
    split = Xtrain.shape[0] * pct // 100
    Xtr = Xtrain.iloc[:split]
    ytr = ytrain.iloc[:split]
    # Тест — фиксированный holdout, не трогаем
    Xte = Xtest
    yte = ytest

    ax = axes[idx]
    ax.plot(yte.values[-tail_n:], label="FACT", color="black", linewidth=2)

    for name, model in pipelines[:5]:
        model.fit(Xtr, ytr)
        ypred = model.predict(Xte)
        ax.plot(ypred[-tail_n:], label=name.replace("Scaled_", ""), alpha=0.85)

    ax.set_title(f"Train={pct}% ({split} строк) | последние {tail_n} тест. точек")
    ax.set_xlabel("Test index (tail)")
    ax.set_ylabel("duration_hours")
    ax.grid(True, alpha=0.25)
    ax.legend(loc=3, ncols=2, fontsize=8)

for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Модели 1-5: предсказания при разном размере train (тест фиксирован)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 28))
axes = axes.flatten()

models_to_plot = pipelines[5:10]

for idx, pct in enumerate(plot_values):
    split = Xtrain.shape[0] * pct // 100
    Xtr = Xtrain.iloc[:split]
    ytr = ytrain.iloc[:split]
    Xte = Xtest
    yte = ytest

    ax = axes[idx]
    ax.plot(yte.values[-tail_n:], label="FACT", linewidth=2, color="black")

    for name, model in models_to_plot:
        model.fit(Xtr, ytr)
        ypred = model.predict(Xte)
        ax.plot(ypred[-tail_n:], label=name.replace("Scaled_", ""), alpha=0.9)

    ax.set_title(f"Train={pct}% ({split} строк) | последние {tail_n} тест. точек")
    ax.set_xlabel("Test index (tail)")
    ax.set_ylabel("duration_hours")
    ax.legend(loc="best", ncols=2, fontsize=8)
    ax.grid(True, alpha=0.25)

for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Модели 6-10: предсказания при разном размере train (тест фиксирован)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 28))
axes = axes.flatten()

for idx, pct in enumerate(plot_values):
    split = Xtrain.shape[0] * pct // 100
    Xtr = Xtrain.iloc[:split]
    ytr = ytrain.iloc[:split]
    Xte = Xtest
    yte = ytest

    ax = axes[idx]
    ax.plot(yte.values[-tail_n:], label="FACT", linewidth=2, color="black")

    for name, model in pipelines[10:]:
        model.fit(Xtr, ytr)
        ypred = model.predict(Xte)
        ax.plot(ypred[-tail_n:], label=name.replace("Scaled_", ""), alpha=0.85)

    ax.set_title(f"Train={pct}% ({split} строк) | последние {tail_n} тест. точек")
    ax.set_xlabel("Test index (tail)")
    ax.set_ylabel("duration_hours")
    ax.grid(True, alpha=0.25)
    ax.legend(loc=3, ncols=3, fontsize=8)

for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Модели 11+: предсказания при разном размере train (тест фиксирован)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Для последующих графиков берём top-7 по CV на типовых задачах,
# а не по holdout, чтобы не подмешивать test в отбор моделей.
best_models = [
    name.replace("Scaled_", "")
    for name in cv_summary["Model"].head(7).tolist()
]

best_models


In [ ]:
plot_values = list(range(10, 100, 10))  # 10, 20, ..., 90
n_plots = len(plot_values)
ncols = 2
nrows = math.ceil(n_plots / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 28))
axes = axes.flatten()

for idx, i in enumerate(plot_values):
    split = Xtrain.shape[0] * i // 100
    Xtr = Xtrain.iloc[:split]
    ytr = ytrain.iloc[:split]
    Xte = Xtest
    yte = ytest

    ax = axes[idx]
    ax.plot(yte.values[-tail_n:], label="FACT", linewidth=2, color="black")

    for name in best_models:
        model = pipelines[[i_p[0] for i_p in pipelines].index('Scaled_' + name)][1]
        model.fit(Xtr, ytr)

        ypred = model.predict(Xte)
        residuals = ypred - yte

        ax.plot(residuals.values[-200:], label=name)

    ax.set_title(f'Размер обучающей выборки: {i}%')
    ax.set_xlabel("Test index (tail)")
    ax.set_ylabel("Residual (pred − fact)")
    ax.legend(loc=3, ncols=3, fontsize=8)
    ax.grid(True, alpha=0.25)

for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Residuals лучших моделей при разном размере train", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
